[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S15_matplotlib_graficos_que_comunican.ipynb)

# Sesión 15 · Matplotlib: gráficos que comunican

**Módulo 4: Matplotlib** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Combinar varios gráficos en una figura con `subplots` y comparar grupos con mini gráficos que comparten eje.
2. Graficar directamente desde pandas.
3. Guiar la lectura con anotaciones, etiquetas selectivas y color con intención.
4. Reconocer y corregir los errores más comunes, y guardar un gráfico en buena resolución.

## 📋 Qué debes saber antes
Sesión 14: `fig`, `ax`, los tipos de gráfico básicos y los formateadores.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el mismo estilo de la sesión 14 (colores `AZUL`, `NARANJA`, `AQUA`... y `GRIS`) y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica un estilo sobrio a los gráficos y carga los verificadores.
import copy
import hashlib
import math
import os
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, MultipleLocator, PercentFormatter, StrMethodFormatter

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: ventas de tiendas ----------
meses = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "set", "oct", "nov", "dic"]
_estacion = np.array([0.9, 0.85, 0.95, 1.0, 1.0, 1.05, 1.1, 1.0, 0.95, 1.0, 1.1, 1.35])
_nivel = {"Barranco": 110_000, "Lince": 150_000, "Miraflores": 205_000, "San Isidro": 180_000, "Surco": 230_000}
_tendencia = {"Barranco": 1.0, "Lince": 0.98, "Miraflores": 1.0, "San Isidro": 1.01, "Surco": 1.035}
ventas_mes_tienda = pd.DataFrame(
    {t: np.round(n * _estacion * _tendencia[t] ** np.arange(12) * rng.uniform(0.95, 1.05, 12), -2) for t, n in _nivel.items()},
    index=meses)
ventas_tienda = ventas_mes_tienda.sum()

# ---------- Datos de práctica: banco ----------
flujo_mensual = pd.Series(np.round(rng.normal(420_000, 60_000, 12), -2), index=meses)
flujo_mensual["may"] = 210_000.0                                   # un mes muy bajo
egresos_canal = pd.Series({"agencia": 1_250_000.0, "app": 2_980_000.0, "cajero": 1_640_000.0, "teléfono": 310_000.0, "web": 870_000.0})
montos_egreso = np.round(rng.lognormal(5.0, 0.8, 800), 2)
reclamos_100 = pd.Series({"Barranco": 140.0, "Lince": 62.5, "Miraflores": 40.0, "San Isidro": 25.0, "Surco": 60.0})


def grafico_confuso():
    """Un gráfico con varios errores comunes, para que lo arregles en el ejercicio 5."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(ventas_tienda.index, ventas_tienda.values, color=[ROJO, AMARILLO, VERDE, VIOLETA, MAGENTA])
    ax.set_ylim(1_000_000, ventas_tienda.max() * 1.02)
    ax.grid(axis="both", linestyle="--")
    return fig, ax


_D = copy.deepcopy({k: globals()[k] for k in ["ventas_mes_tienda", "ventas_tienda", "flujo_mensual", "egresos_canal",
                                               "montos_egreso", "reclamos_100"]})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _barras_h(r, nombre, ax, serie, destacar=None):
    """Revisa barras horizontales ordenadas con la mayor arriba y, si se pide, el énfasis de color."""
    barras = _barras(ax)
    orden = sorted(serie.items(), key=lambda kv: kv[1])
    if len(barras) != len(serie):
        r.mal(f"`{nombre}` tiene {len(barras)} barras y se esperaban {len(serie)}.")
        return
    if not _cerca_lista([b.get_width() for b in barras], [v for _, v in orden]):
        r.mal(f"Las barras de `{nombre}` deberían ser horizontales y estar ordenadas con la mayor arriba.")
        return
    colores = [_hex(b.get_facecolor()) for b in barras]
    if destacar is None:
        if len(set(colores)) != 1:
            r.mal(f"Las barras de `{nombre}` deberían ir en un solo color.")
            return
    else:
        esperado = [AZUL if k == destacar else GRIS for k, _ in orden]
        if colores != esperado:
            r.mal(f"En `{nombre}`, solo la barra de {destacar} debería ir en `AZUL` y las demás en `GRIS`.")
            return
    r.ok(f"Las barras de `{nombre}` son correctas.")


def _suptitulo(fig):
    t = getattr(fig, "_suptitle", None)
    return t.get_text() if t is not None else ""


def check_ejercicio_1():
    r = _Revision("Ejercicio 1")
    vt, vmt = _D["ventas_tienda"], _D["ventas_mes_tienda"]
    axs = r.var("axs1")
    if axs is not _FALTA:
        if not isinstance(axs, np.ndarray) or axs.shape != (2,):
            r.mal("`axs1` debería ser el array de 2 ejes que devuelve `plt.subplots(1, 2)`.")
        else:
            _barras_h(r, "axs1[0]", axs[0], vt)
            if _formato(axs[0], "x", 2_500_000) != "S/ 2.5 M":
                r.mal("En `axs1[0]`, el eje de valores debería verse en millones, como \"S/ 2.5 M\".")
            lineas = axs[1].get_lines()
            totales = [sum(fila) for fila in vmt.values.tolist()]
            if len(lineas) == 1 and _cerca_lista(lineas[0].get_ydata(), totales):
                r.ok("La línea de `axs1[1]` tiene el total de cada mes.")
            else:
                r.mal("`axs1[1]` debería tener una línea con la venta total de cada mes (la suma de las tiendas).")
            _rotulos(r, "axs1[0]", axs[0], "Ventas por tienda")
            _rotulos(r, "axs1[1]", axs[1], "Ventas totales por mes")
    fig = r.var("fig1")
    if fig is not _FALTA and isinstance(fig, mpl.figure.Figure):
        r.ok("La figura tiene el título general pedido.") if _texto(_suptitulo(fig), "Resumen de ventas 2025") else \
            r.mal("Falta el título general de `fig1` (investiga `fig.suptitle`), con el texto pedido.")
    axs = r.var("axs2")
    if axs is not _FALTA:
        if not isinstance(axs, np.ndarray) or axs.shape != (5,):
            r.mal("`axs2` debería ser el array de 5 ejes de `plt.subplots(1, 5, ...)`.")
        else:
            bien = all(len(ax.get_lines()) == 1 and _cerca_lista(ax.get_lines()[0].get_ydata(), vmt[t].tolist())
                       and _texto(ax.get_title(loc="left") or ax.get_title(), t) for ax, t in zip(axs, vmt.columns))
            r.ok("Cada mini gráfico de `axs2` tiene su tienda.") if bien else \
                r.mal("Cada eje de `axs2` debería tener una línea con las ventas de una tienda, en el orden de las columnas, y el nombre de la tienda como título.")
            r.ok("Los mini gráficos comparten el eje y.") if axs[0].get_shared_y_axes().joined(axs[0], axs[4]) else \
                r.mal("Los mini gráficos de `axs2` deberían compartir el eje y (`sharey=True`) para poder compararlos.")
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2")
    vmt = _D["ventas_mes_tienda"]
    ax = _grafico(r, "ax3")
    if ax is not None:
        lineas = ax.get_lines()
        leyenda = ax.get_legend()
        if len(lineas) != 5 or not all(_cerca_lista(l.get_ydata(), vmt[t].tolist()) for l, t in zip(lineas, vmt.columns)):
            r.mal("`ax3` debería tener una línea por tienda, dibujadas desde `ventas_mes_tienda` con `.plot()`.")
        elif leyenda is None or [t.get_text() for t in leyenda.get_texts()] != list(vmt.columns):
            r.mal("`ax3` debería tener una leyenda con el nombre de cada tienda.")
        elif _barras(ax):
            r.mal("En `ax3` hay barras mezcladas con las líneas: dibuja las barras de `ax4` en su propio eje (`ax=ax4`).")
        else:
            r.ok("`ax3` tiene las 5 líneas con su leyenda.")
        _rotulos(r, "ax3", ax, "Ventas mensuales por tienda")
    ax = _grafico(r, "ax4")
    if ax is not None:
        if ax is globals().get("ax3") or ax.get_lines():
            r.mal("`ax4` debería ser un gráfico aparte, solo con barras: créalo con `plt.subplots()` y pásalo con `ax=ax4`.")
        else:
            _barras_h(r, "ax4", ax, _D["ventas_tienda"])
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    vmt = _D["ventas_mes_tienda"]
    totales = [sum(fila) for fila in vmt.values.tolist()]
    ax = _grafico(r, "ax5")
    if ax is not None:
        notas = [t for t in ax.texts if isinstance(t, mpl.text.Annotation)]
        mayor = max(totales)
        if not notas:
            r.mal("A `ax5` le falta una anotación (`ax.annotate`).")
        elif not any(_texto(n.get_text(), "Campaña navideña") for n in notas):
            r.mal("La anotación de `ax5` debería decir \"Campaña navideña\".")
        elif not any(abs(float(n.xy[0]) - totales.index(mayor)) < 1e-6 and abs(float(n.xy[1]) - mayor) < 1e-6 for n in notas):
            r.mal("La flecha de la anotación debería apuntar al mes de mayor venta: su posición en x y su valor en y.")
        elif not any(n.arrow_patch is not None for n in notas):
            r.mal("La anotación debería tener una flecha (`arrowprops`).")
        else:
            r.ok("La anotación de `ax5` es correcta.")
    ax = _grafico(r, "ax6")
    if ax is not None:
        lineas = ax.get_lines()
        leyenda = ax.get_legend()
        textos = [t.get_text() for t in ax.texts]
        finales = {t: vmt[t].tolist()[-1] for t in ("Surco", "Lince")}
        esperados = [f"S/ {v / 1000:.0f} mil" for v in finales.values()]
        if len(lineas) != 2:
            r.mal("`ax6` debería tener 2 líneas: Surco y Lince.")
        elif leyenda is None or sorted(t.get_text() for t in leyenda.get_texts()) != ["Lince", "Surco"]:
            r.mal("`ax6` debería tener una leyenda con Surco y Lince.")
        elif not all(e in textos for e in esperados):
            r.mal("Falta etiquetar el último valor de cada línea con el formato \"S/ 245 mil\" (usa `ax.text`).")
        elif len(textos) > 2:
            r.mal("En `ax6` hay más etiquetas de las pedidas: etiqueta solo el último valor de cada línea.")
        else:
            r.ok("`ax6` tiene leyenda y etiqueta solo el último valor de cada línea.")
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    ax = _grafico(r, "ax7")
    if ax is not None:
        _barras_h(r, "ax7", ax, _D["ventas_tienda"], destacar="Miraflores")
    ax = _grafico(r, "ax8")
    if ax is not None:
        vmt = _D["ventas_mes_tienda"]
        lineas = ax.get_lines()
        if len(lineas) != 5:
            r.mal(f"`ax8` tiene {len(lineas)} líneas y se esperaban 5, una por tienda.")
        else:
            azules = [l for l in lineas if _hex(l.get_color()) == AZUL]
            grises = [l for l in lineas if _hex(l.get_color()) == GRIS]
            if len(azules) != 1 or len(grises) != 4:
                r.mal("En `ax8`, una sola línea debería ir en `AZUL` y las otras cuatro en `GRIS`.")
            elif not _cerca_lista(azules[0].get_ydata(), vmt["Surco"].tolist()):
                r.mal("La línea destacada de `ax8` debería ser la de Surco.")
            elif not azules[0].get_linewidth() > max(l.get_linewidth() for l in grises):
                r.mal("La línea de Surco debería ser más gruesa que las grises.")
            elif "Surco" not in [t.get_text() for t in ax.texts]:
                r.mal("Etiqueta la línea destacada con el texto \"Surco\" junto a su último punto.")
            else:
                r.ok("`ax8` destaca a Surco y deja las demás tiendas de contexto.")
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    vt = _D["ventas_tienda"]
    ax = _grafico(r, "ax9")
    if ax is not None:
        _barras_h(r, "ax9", ax, vt)
        izquierda = ax.get_xlim()[0]
        r.ok("El eje de valores empieza en 0.") if abs(izquierda) < 1e-9 else \
            r.mal(f"El eje de valores de `ax9` empieza en {izquierda:,.0f}; en un gráfico de barras debe empezar en 0.")
        lider = max(vt.items(), key=lambda kv: kv[1])[0]
        titulo = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
        if not titulo:
            r.mal("A `ax9` le falta un título que cuente la conclusión.")
        elif lider.lower() not in titulo.lower():
            r.mal("El título de `ax9` debería contar la conclusión: mencionar qué tienda vendió más.")
        else:
            r.ok("El título de `ax9` cuenta la conclusión.")
        etiqueta = _formato(ax, "x", 2_500_000)
        r.ok("El eje de valores está en millones.") if etiqueta == "S/ 2.5 M" else \
            r.mal(f"En `ax9`, 2500000 se muestra como {etiqueta!r} y debería verse como \"S/ 2.5 M\".")
        _rotulos(r, "ax9", ax, None, "Ventas (S/)")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_torta": "bcdd8aab86ee4cff5c81786a338d0b132b6b9a4e26232a31b154265103dd3259",
        "pred_doble_eje": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
        "pred_barras_cero": "6c5fb3b25e6ba7dcf12155440e0c51b36a0492e24123968a10f8c31c304ebb6e",
    })
    r.fin()


def _png(r, ruta, ancho_min):
    if not os.path.exists(ruta):
        r.mal(f"No encuentro el archivo `{ruta}`. ¿Llamaste a `savefig` con ese nombre?")
        return
    alto, ancho = mpimg.imread(ruta).shape[:2]
    r.ok(f"`{ruta}` existe y tiene buena resolución ({ancho} × {alto} píxeles).") if ancho >= ancho_min else \
        r.mal(f"`{ruta}` mide {ancho} píxeles de ancho: con `dpi=200` debería superar los {ancho_min}.")


def check_ejercicio_6():
    r = _Revision("Ejercicio 6")
    _png(r, "ventas_tiendas.png", 1200)
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    axs = r.var("axs_lam")
    if axs is not _FALTA:
        if not isinstance(axs, np.ndarray) or axs.shape != (2, 2):
            r.mal("`axs_lam` debería ser la matriz de 2 × 2 ejes de `plt.subplots(2, 2, ...)`.")
        else:
            f = _D["flujo_mensual"]
            ax = axs[0, 0]
            linea = ax.get_lines()
            notas = [t for t in ax.texts if isinstance(t, mpl.text.Annotation)]
            peor = f.tolist().index(min(f.tolist()))
            if not linea or not _cerca_lista(linea[0].get_ydata(), f.tolist()):
                r.mal("`axs_lam[0, 0]` debería tener la línea del flujo mensual.")
            elif not any(abs(float(n.xy[0]) - peor) < 1e-6 and abs(float(n.xy[1]) - min(f.tolist())) < 1e-6 for n in notas):
                r.mal("En `axs_lam[0, 0]`, anota el mes de menor flujo (la flecha debe apuntar a ese punto).")
            elif _formato(ax, "y", 400_000) != "S/ 400 mil":
                r.mal("En `axs_lam[0, 0]`, el eje y debería verse en miles, como \"S/ 400 mil\".")
            else:
                r.ok("`axs_lam[0, 0]` es correcto.")
            eg = _D["egresos_canal"]
            _barras_h(r, "axs_lam[0, 1]", axs[0, 1], eg, destacar=max(eg.items(), key=lambda kv: kv[1])[0])
            if _formato(axs[0, 1], "x", 2_500_000) != "S/ 2.5 M":
                r.mal("En `axs_lam[0, 1]`, el eje de valores debería verse en millones, como \"S/ 2.5 M\".")
            ax = axs[1, 0]
            barras = _barras(ax)
            mediana = statistics.median(_D["montos_egreso"].tolist())
            verticales = [l for l in ax.get_lines() if len(set(map(float, l.get_xdata()))) == 1]
            if len(barras) != 30 or round(sum(b.get_height() for b in barras)) != len(_D["montos_egreso"]):
                r.mal("`axs_lam[1, 0]` debería tener el histograma de `montos_egreso` con 30 intervalos.")
            elif not any(abs(float(l.get_xdata()[0]) - mediana) < 1e-6 for l in verticales):
                r.mal("En `axs_lam[1, 0]`, marca la mediana con una línea vertical.")
            else:
                r.ok("`axs_lam[1, 0]` es correcto.")
            rc = _D["reclamos_100"]
            _barras_h(r, "axs_lam[1, 1]", axs[1, 1], rc, destacar=max(rc.items(), key=lambda kv: kv[1])[0])
            for nombre, ax, titulo in zip(["axs_lam[0, 0]", "axs_lam[0, 1]", "axs_lam[1, 0]", "axs_lam[1, 1]"], axs.flat,
                                          ["Flujo mensual", "Egresos por canal", "Monto de los egresos", "Reclamos por cada 100 clientes"]):
                _rotulos(r, nombre, ax, titulo)
    fig = r.var("fig_lam")
    if fig is not _FALTA and isinstance(fig, mpl.figure.Figure):
        r.ok("La lámina tiene su título general.") if _suptitulo(fig).strip() else r.mal("Falta el título general de la lámina (`suptitle`).")
    _png(r, "lamina_banco.png", 1800)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    ax = _grafico(r, "ax_mapa")
    if ax is not None:
        mallas = [c for c in ax.collections if isinstance(c, mpl.collections.QuadMesh)]
        tabla = _D["ventas_mes_tienda"].T / 1000
        if not mallas:
            r.mal("`ax_mapa` debería ser un mapa de calor de seaborn (`sns.heatmap`).")
        else:
            valores = np.asarray(mallas[0].get_array()).ravel().tolist()
            esperado = [x for fila in tabla.values.tolist() for x in fila]
            r.ok("El mapa de calor muestra las ventas en miles por tienda y mes.") if _cerca_lista(valores, esperado, 1e-6) else \
                r.mal("El mapa de calor debería tener las tiendas en filas, los meses en columnas y las ventas en miles.")
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
- `ventas_mes_tienda`: ventas de cada mes (filas) y tienda (columnas) en 2025, y `ventas_tienda`: el total anual de cada tienda.
- `flujo_mensual`: el flujo neto mensual de un banco, `egresos_canal`: egresos por canal, `montos_egreso`: 800 egresos individuales y `reclamos_100`: reclamos por cada 100 clientes en cada distrito.
- `grafico_confuso()`: una función que dibuja un gráfico con errores, para el ejercicio 5.

In [ ]:
print(ventas_mes_tienda.head(3), "\n")
print(ventas_tienda, "\n")
print(egresos_canal, "\n")
print(reclamos_100)

---
## 1. Varios gráficos en una figura: `subplots`

### 📘 Concepto
`plt.subplots(filas, columnas)` crea una figura con varios ejes y los devuelve en un **array**: con `(1, 2)` son `axs[0]` y `axs[1]`; con `(2, 2)`, `axs[0, 0]` a `axs[1, 1]`.
- `fig.suptitle("...")` pone un título general a toda la figura.
- `fig.tight_layout()` ajusta los espacios para que nada se encime.
- Con `sharey=True`, todos los ejes usan la **misma escala en y**. Así, una fila de mini gráficos (uno por grupo) se compara de un vistazo. Es la alternativa a meter muchas líneas en un solo gráfico, y la forma correcta de mostrar dos medidas distintas: dos gráficos, **nunca dos ejes y** en uno.

In [ ]:
gen_ej = np.random.default_rng(3)
fig_ej, axs_ej = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, nombre in zip(axs_ej, ["Norte", "Centro", "Sur"]):
    ax.plot(range(1, 7), gen_ej.integers(20, 60, 6))
    ax.set_title(nombre)
fig_ej.suptitle("Pedidos por mes en cada zona")
fig_ej.tight_layout()

### ✍️ Tu turno · Ejercicio 1: resumen en dos vistas y mini gráficos
1. `fig1, axs1`: una figura de 1 × 2 (tamaño 11 × 4) con:
   - `axs1[0]`: barras horizontales con `ventas_tienda`, la mayor arriba, título `Ventas por tienda` y el eje de valores en millones con 1 decimal (`S/ 2.5 M`), con `FuncFormatter(lambda x, pos: f"S/ {x / 1_000_000:.1f} M")`.
   - `axs1[1]`: una línea con la venta total de cada mes (suma de las tiendas), título `Ventas totales por mes`.
   - Título general: `Resumen de ventas 2025`. Termina con `fig1.tight_layout()`.
2. `fig2, axs2`: una fila de 5 mini gráficos (tamaño 15 × 3) que **compartan el eje y**, cada uno con la línea mensual de una tienda (en el orden de las columnas de `ventas_mes_tienda`) y el nombre de la tienda como título. Para que los meses no se encimen, deja una marca cada 3 meses con `ax.set_xticks(range(0, 12, 3))`.

¿Qué tienda crece a lo largo del año y cuál cae?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

La venta total por mes es `ventas_mes_tienda.sum(axis=1)`. Para los mini gráficos, recorre `zip(axs2, ventas_mes_tienda.columns)`.
</details>

<details><summary>💡 Pista 2</summary>

`fig2, axs2 = plt.subplots(1, 5, figsize=(15, 3), sharey=True)`; dentro del bucle, `ax.plot(ventas_mes_tienda.index, ventas_mes_tienda[t])` y `ax.set_title(t)`.
</details>

---
## 2. Graficar desde pandas

### 📘 Concepto
Las Series y los DataFrames tienen un método `.plot()` que usa Matplotlib por debajo y **devuelve el eje**, así que después puedes seguir personalizándolo como siempre:

```python
ax = df.plot(figsize=(9, 4))            # una línea por columna, con leyenda
ax = serie.plot(kind="barh")            # barras horizontales
ax.set_title("...")
```

`kind` puede ser `"line"`, `"bar"`, `"barh"`, `"hist"`, `"box"`, `"scatter"`... El índice va al eje x y cada columna es una serie.

⚠️ Si no le dices dónde dibujar, una **Series** dibuja sobre el último gráfico abierto y se mezcla con él. La forma segura es crear el eje con `plt.subplots()` y pasárselo: `serie.plot(kind="barh", ax=ax)`.

Ojo: con cinco o más líneas en un mismo gráfico, cuesta seguir cada una (el gráfico "de espagueti"). Para comparar muchas series, los mini gráficos o el énfasis de la sección 4 funcionan mejor.

In [ ]:
pedidos_ej = pd.DataFrame({"app": [120, 150, 170, 210], "tienda": [300, 280, 290, 260]}, index=["T1", "T2", "T3", "T4"])
ax_ej = pedidos_ej.plot(figsize=(6, 3), marker="o")
ax_ej.set_title("Pedidos por canal y trimestre")

fig_ej2, ax_ej2 = plt.subplots(figsize=(6, 2))
pedidos_ej.sum().sort_values().plot(kind="barh", ax=ax_ej2)       # dibuja en el eje que le pasamos
ax_ej2.set_title("Pedidos del año por canal")

### ✍️ Tu turno · Ejercicio 2: pandas dibuja
1. `ax3`: las ventas mensuales de todas las tiendas, con `ventas_mes_tienda.plot(...)` (tamaño 9 × 4). Título `Ventas mensuales por tienda`. ¿Se lee bien?
2. `fig4, ax4`: el total anual de cada tienda en barras horizontales con la mayor arriba, dibujadas con `.plot(kind="barh", ax=ax4)` desde `ventas_tienda`, en su propia figura.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

`ax3 = ventas_mes_tienda.plot(figsize=(9, 4))` crea su propia figura porque es un DataFrame. Para `ax4`, crea antes la figura con `plt.subplots` y pásale el eje a pandas.
</details>

<details><summary>💡 Pista 2</summary>

`fig4, ax4 = plt.subplots(figsize=(7, 4))` y luego `ventas_tienda.sort_values().plot(kind="barh", ax=ax4)`.
</details>

---
## 3. Anotaciones y etiquetas selectivas

### 📘 Concepto
Un buen gráfico señala lo importante:
- **`ax.annotate(texto, xy=(x, y), xytext=(x2, y2), arrowprops=dict(arrowstyle="->"))`** escribe `texto` en `xytext` con una flecha que apunta a `xy`. En un eje con categorías (meses, tiendas), la primera categoría está en la posición 0, la segunda en la 1, etc.
- **`ax.text(x, y, texto)`** escribe un texto en una posición, sin flecha.
- **Etiquetas selectivas**: rotula solo lo que importa (el último valor, el máximo, la serie de interés). Un número en cada punto no se lee.
- Con dos o más series, la **leyenda** siempre va (`label=` en cada serie y `ax.legend()`), aunque además rotules algún valor.

In [ ]:
trimestres_ej = ["T1", "T2", "T3", "T4"]
app_ej, tienda_ej = [120, 150, 170, 260], [300, 280, 290, 250]

fig_ej, ax_ej = plt.subplots(figsize=(7, 3.5))
ax_ej.plot(trimestres_ej, app_ej, marker="o", label="app")
ax_ej.plot(trimestres_ej, tienda_ej, marker="o", label="tienda")
ax_ej.legend()
ax_ej.annotate("Lanzamiento de la app 2.0", xy=(3, 260), xytext=(0.5, 230), arrowprops=dict(arrowstyle="->", color=TINTA_2))
ax_ej.text(3.1, tienda_ej[-1], f"{tienda_ej[-1]}", va="center")    # solo el último valor
ax_ej.set_xlim(-0.3, 3.6)

### ✍️ Tu turno · Ejercicio 3: señalar lo importante
1. `fig5, ax5`: una línea con la venta total de cada mes y una anotación con el texto `Campaña navideña` cuya flecha apunte al mes de **mayor** venta (en x, la posición del mes; en y, su valor).
2. `fig6, ax6`: las líneas mensuales de Surco y Lince, con leyenda y una etiqueta **solo en el último valor** de cada línea, con el formato `S/ 245 mil` (usa `ax6.text` junto al último punto).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

La posición del mes de mayor venta es `total.values.argmax()`, y su valor, `total.max()`, con `total = ventas_mes_tienda.sum(axis=1)`.
</details>

<details><summary>💡 Pista 2</summary>

Para `ax6`, recorre `["Surco", "Lince"]`: dibuja cada línea con `label=t` y escribe `ax6.text(11.2, serie.iloc[-1], f"S/ {serie.iloc[-1] / 1000:.0f} mil", va="center")`. Al final, `ax6.legend()`.
</details>

---
## 4. Color con intención

### 📘 Concepto
El color debe **decir algo**, no decorar:
- **Una serie, un color.** Barras de muchos colores obligan a buscar una leyenda que no existe.
- **Énfasis**: si el mensaje es sobre un elemento, píntalo con un color fuerte (`AZUL`) y deja el resto en `GRIS`. El ojo va directo a lo importante y el resto queda como contexto.
- **Color por identidad, no por posición**: si Surco es azul en un gráfico, que sea azul en todos.
- El rojo y el verde no se usan para series comunes: se reservan para "malo" y "bueno", y siempre acompañados de un texto (casi 1 de cada 12 hombres no distingue bien el rojo del verde).

Con `ax.barh(..., color=lista)` pasas un color por barra, y con `ax.plot(..., color=..., linewidth=...)` controlas cada línea.

In [ ]:
productos_ej = pd.Series({"gorra": 90, "jean": 150, "casaca": 210, "polo": 320})
colores_ej = [AZUL if p == "casaca" else GRIS for p in productos_ej.index]
fig_ej, ax_ej = plt.subplots(figsize=(6, 3))
ax_ej.barh(productos_ej.index, productos_ej.values, color=colores_ej)
ax_ej.set_title("La casaca ya es el segundo producto más vendido")

### ✍️ Tu turno · Ejercicio 4: destacar lo que importa
1. `fig7, ax7`: barras horizontales con `ventas_tienda` (la mayor arriba), con **Miraflores** en `AZUL` y el resto en `GRIS`.
2. `fig8, ax8`: las 5 líneas mensuales de las tiendas, con **Surco** en `AZUL` y grosor 2.5, las otras cuatro en `GRIS` y grosor 1, y el texto `Surco` junto al último punto de su línea. Ponle un título que cuente qué pasa con Surco.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Arma la lista de colores recorriendo el índice ya ordenado: `[AZUL if t == "Miraflores" else GRIS for t in orden.index]`.
</details>

<details><summary>💡 Pista 2</summary>

En `ax8`, recorre las columnas y elige color y grosor con una expresión condicional. Dibuja Surco al final para que quede por encima.
</details>

---
## 5. Un gráfico, una conclusión: errores comunes

### 📘 Concepto
Cada gráfico debe responder **una** pregunta, y el título puede contar la respuesta ("Surco lidera las ventas del año") en vez de solo describir ("Ventas por tienda"). Errores frecuentes:

| Error | Por qué engaña o confunde | Qué hacer |
|---|---|---|
| Barras que no empiezan en 0 | exageran las diferencias | eje desde 0 |
| Un color por barra sin motivo | parece que el color significa algo | un solo color, o énfasis |
| Barras sin ordenar | obliga a comparar a ojo | ordenar por valor |
| Torta con muchas porciones parecidas | el ojo no compara bien ángulos | barras |
| Dos ejes y en un gráfico | la alineación de las escalas es arbitraria e inventa relaciones | dos gráficos, o índices con la misma base |
| Grilla gruesa o punteada, sin título ni unidades | ruido y dudas | grilla tenue, título y unidades |
| Un número en cada punto | nadie los lee | etiquetas selectivas |

In [ ]:
fig_mal, ax_mal = grafico_confuso()     # mira todo lo que está mal

### ✍️ Tu turno · Ejercicio 5: arreglar el gráfico confuso
**Parte A.** Rehaz el gráfico de `grafico_confuso()` como `fig9, ax9`:
- barras horizontales con `ventas_tienda`, la mayor arriba, en un solo color;
- eje de valores desde 0, en millones con 1 decimal (`S/ 2.5 M`), con la etiqueta `Ventas (S/)`;
- un título que cuente la conclusión: qué tienda vendió más en el año.

**Parte B.** Responde:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_torta` | para comparar las ventas de 7 productos con valores parecidos, ¿torta o barras? | `"torta"` o `"barras"` |
| `pred_doble_eje` | ¿conviene mostrar ventas y temperatura con dos ejes y en un mismo gráfico? | `"sí"` o `"no"` |
| `pred_barras_cero` | ¿el eje de un gráfico de barras debe empezar en 0? | `"sí"` o `"no"` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Empieza de cero con `fig9, ax9 = plt.subplots(...)`; no reutilices `fig_mal`.
</details>

<details><summary>💡 Pista 2</summary>

`ax9.set_xlim(left=0)` fija el inicio del eje. El formateador de millones es el mismo del ejercicio 1.
</details>

---
## 6. Guardar en buena resolución: `savefig`

### 📘 Concepto
`fig.savefig("archivo.png", dpi=200, bbox_inches="tight")` guarda la figura:
- `dpi` son los puntos por pulgada: con 200, una figura de 8 pulgadas de ancho mide unos 1600 píxeles, suficiente para un informe o una publicación.
- `bbox_inches="tight"` recorta los márgenes sobrantes sin cortar títulos ni etiquetas.
- El formato sale de la extensión: `.png` para publicar y `.svg` o `.pdf` si necesitas vectores.

En Colab, el archivo queda en la carpeta de la sesión (panel 📁 de la izquierda), desde donde puedes descargarlo.

In [ ]:
fig_ej, ax_ej = plt.subplots(figsize=(4, 2))
ax_ej.bar(["a", "b"], [3, 5])
fig_ej.savefig("prueba_ej.png", dpi=200, bbox_inches="tight")
print(mpimg.imread("prueba_ej.png").shape)      # (alto, ancho, canales) en píxeles

### ✍️ Tu turno · Ejercicio 6: exportar
Guarda `fig9` (el gráfico arreglado) como `ventas_tiendas.png` con 200 dpi y sin márgenes sobrantes.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

Una sola línea: `fig9.savefig(...)`.
</details>

<details><summary>💡 Pista 2</summary>

Los parámetros son `dpi=200` y `bbox_inches="tight"`.
</details>

---
## 🏋️ Reto final: una lámina para gerencia
Crea `fig_lam, axs_lam` con 2 × 2 gráficos (tamaño 12 × 8), cada uno con su título:
1. `axs_lam[0, 0]` · `Flujo mensual`: una línea con `flujo_mensual`, el eje y en miles (`S/ 400 mil`) y una anotación cuya flecha apunte al mes de **menor** flujo.
2. `axs_lam[0, 1]` · `Egresos por canal`: barras horizontales de `egresos_canal`, la mayor arriba, con el canal de mayor egreso en `AZUL` y el resto en `GRIS`, y el eje en millones (`S/ 2.5 M`) con una marca por millón (`MultipleLocator(1_000_000)`).
3. `axs_lam[1, 0]` · `Monto de los egresos`: histograma de `montos_egreso` en 30 intervalos, con una línea vertical en la **mediana**.
4. `axs_lam[1, 1]` · `Reclamos por cada 100 clientes`: barras horizontales de `reclamos_100`, la mayor arriba, con el distrito de más reclamos en `AZUL` y el resto en `GRIS`.

Agrega un título general, ajusta los espacios con `tight_layout` y guárdala como `lamina_banco.png` con 200 dpi.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Resuelve un gráfico a la vez, sobre su eje (`ax = axs_lam[0, 0]`...). Reutiliza el código de los ejercicios anteriores.
</details>

<details><summary>💡 Pista 2</summary>

Para la anotación del mínimo: `flujo_mensual.values.argmin()` da la posición en x y `flujo_mensual.min()`, el valor en y. La mediana es `np.median(montos_egreso)`.
</details>

---
## 🚀 Nivel pro (opcional): seaborn
**seaborn** es una librería construida sobre Matplotlib, con gráficos estadísticos listos (ya viene instalada en Colab). Crea `ax_mapa` con `sns.heatmap` de las ventas **en miles**, con las tiendas en las filas y los meses en las columnas, usando una escala de un solo color (`cmap="Blues"`: más oscuro, más venta). Pista: `import seaborn as sns` y transpone `ventas_mes_tienda`. ¿En qué meses se ven las celdas más oscuras?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Crear una figura con varios gráficos y darle un título general.
- [ ] Usar mini gráficos con `sharey=True` para comparar grupos.
- [ ] Explicar por qué no se usan dos ejes y en un gráfico.
- [ ] Graficar desde pandas y seguir personalizando el eje que devuelve.
- [ ] Anotar el punto clave de un gráfico y etiquetar solo lo importante.
- [ ] Destacar un elemento con color y dejar el resto en gris.
- [ ] Escribir un título que cuente la conclusión.
- [ ] Guardar un gráfico con `savefig` en buena resolución.

**Próxima sesión (S16):** cierre del análisis exploratorio del proyecto y la primera publicación.